In [1]:
# importing necessary libraries
import numpy as np
import pandas as pd
import os
import glob

# import matplotlib.pyplot as plt
# import seaborn as sns

In [2]:
import sys
print(sys.version)

3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]


In [3]:
# ============================================================
# DATA DICTIONARY (from dataset documentation)
#
# ejection_fraction   (LVEF)  %     normal 55 - 70
# lv_diameter         (LVEDD) cm    normal 3.5 - 5.6
# mitral_e_velocity   (E)     m/s
# mitral_a_velocity   (A)     m/s
# e_a_ratio           (E/A)         normal 0.6 - 1.32
# tricuspid_velocity          m/s
# tricuspid_pressure          mmHg
#
# Values outside the normal range are NOT treated as dirty data.
# This is a heart failure dataset, so abnormal values are expected.
# Only impossible values or unit mix-ups get changed.
# ============================================================

In [4]:
# helper: normal-range flag that stays blank (<NA>) when the value is missing,
# so "no data" is never counted as "not normal"
def range_flag(series, low, high):
    return series.between(low, high).astype("boolean").mask(series.isna())

In [5]:
DATA_DIR = "data"

csv_files = glob.glob(os.path.join(DATA_DIR, "*.csv"))

csv_files

['data\\cardiac_complications.csv',
 'data\\demography.csv',
 'data\\hospitalization_discharge.csv',
 'data\\labs.csv',
 'data\\patienthistory.csv',
 'data\\patient_precriptions.csv',
 'data\\responsivenes.csv']

In [6]:
# Confirming the total number of csv files found
print("Number of CSV files found:", len(csv_files))

Number of CSV files found: 7


In [7]:
# Count rows in each CSV in the dataset
for file in csv_files:
    temp = pd.read_csv(file)
    print(os.path.basename(file), "->", len(temp), "rows")

# os.path.basename extracts the file name from the full file path
# Total count of rows per csv file helps to compare the data at later stages for cleaning and analysis
# It also verifies the total csv files count along with the number of rows

cardiac_complications.csv -> 2008 rows
demography.csv -> 2009 rows
hospitalization_discharge.csv -> 2008 rows
labs.csv -> 2008 rows
patienthistory.csv -> 2008 rows
patient_precriptions.csv -> 15362 rows
responsivenes.csv -> 2008 rows


In [8]:
## Separate the columns and quick preview

csv_check = pd.read_csv(csv_files[0]).head(50)

csv_check

# To understand the structure of dataset before data cleaning, we used two functions
# read_csv to read the csv data into a python dataframe
# head() will show the top few records for preview

,inpatient_number,nyha_cardiac_function_classification,killip_grade,myocardial_infarction,congestive_heart_failure,peripheral_vascular_disease,type_of_heart_failure,lvef,left_ventricular_end_diastolic_diameter_lv,mitral_valve_ems,mitral_valve_ams,ea,tricuspid_valve_return_velocity,tricuspid_valve_return_pressure
0,848044,3,1,0,0,0,Both,NaN,41.0,1.00,NaN,NaN,NaN,NaN
1,821472,3,1,0,1,0,Both,NaN,42.0,1.18,NaN,NaN,2.90,33.0
2,866373,3,2,0,1,0,Both,NaN,48.0,0.59,NaN,NaN,2.60,57.0
3,793147,3,3,0,0,0,Left,NaN,53.0,NaN,NaN,NaN,2.20,19.0
4,739615,3,2,0,0,0,Both,NaN,68.0,0.38,0.58,NaN,NaN,NaN
5,817939,3,2,0,0,0,Both,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,760822,3,1,0,0,0,Both,NaN,54.0,1.54,NaN,NaN,3.00,36.0
7,831189,3,3,0,0,0,Both,NaN,55.0,0.94,1.12,NaN,NaN,NaN
8,837384,2,1,0,0,0,Both,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,798240,3,3,0,0,0,Both,NaN,53.0,NaN,NaN,NaN,2.90,35.0


In [9]:
# To read the CSV file into the original variable
data = pd.read_csv(os.path.join(DATA_DIR, "cardiac_complications.csv"))

In [10]:
# Create a independent copy for cleaning
df = data.copy()

In [11]:
# Preview the content of the Data file
print(df.head())
print(df.shape)
print(df.columns)
print(df.tail())

   inpatient_number  nyha_cardiac_function_classification  killip_grade  \
0            848044                                     3             1   
1            821472                                     3             1   
2            866373                                     3             2   
3            793147                                     3             3   
4            739615                                     3             2   

   myocardial_infarction  congestive_heart_failure  \
0                      0                         0   
1                      0                         1   
2                      0                         1   
3                      0                         0   
4                      0                         0   

   peripheral_vascular_disease type_of_heart_failure  lvef  \
0                            0                  Both   NaN   
1                            0                  Both   NaN   
2                            0        

In [12]:
# ============================================================
# STEP 1: RENAME COLUMNS
#
# Descriptive column names that aren't too long
# ============================================================
df = df.rename(columns={
    "inpatient_number": "patient_id",
    "nyha_cardiac_function_classification": "nyha_class",
    "killip_grade": "killip_grade",
    "myocardial_infarction": "heart_attack",
    "congestive_heart_failure": "congestive_hf",
    "peripheral_vascular_disease": "peripheral_vasc_disease",
    "type_of_heart_failure": "hf_type",
    "lvef": "ejection_fraction",
    "left_ventricular_end_diastolic_diameter_lv": "lv_diameter",
    "mitral_valve_ems": "mitral_e_velocity",
    "mitral_valve_ams": "mitral_a_velocity",
    "ea": "e_a_ratio",
    "tricuspid_valve_return_velocity": "tricuspid_velocity",
    "tricuspid_valve_return_pressure": "tricuspid_pressure",
})

print(df.shape)

(2008, 14)


In [13]:
# ============================================================
# STEP 2: REMOVE DUPLICATE ROWS
# ============================================================
duplicate_count = df.duplicated().sum()
print("\nDuplicate rows found:", duplicate_count)
df = df.drop_duplicates().copy()

duplicate_ids = df["patient_id"].duplicated().sum()
print("Repeated patient IDs found:", duplicate_ids)
if duplicate_ids > 0:
    print(df[df["patient_id"].duplicated(keep=False)].sort_values("patient_id"))



Duplicate rows found: 0
Repeated patient IDs found: 0


In [14]:
# ============================================================
# STEP 3: CLEAN TEXT VALUES
# ============================================================
df["hf_type"] = df["hf_type"].astype("string").str.strip().str.title()
print("\nhf_type values:", df["hf_type"].value_counts().to_dict())


hf_type values: {'Both': 1480, 'Left': 477, 'Right': 51}


In [15]:
# ============================================================
# STEP 4: CLEAN CATEGORY COLUMNS
#
# NYHA class: 1-4, Killip grade: 1-4, disease columns: 0 = No, 1 = Yes
# Anything else becomes missing.
# ============================================================
category_rules = {
    "nyha_class": [1, 2, 3, 4],
    "killip_grade": [1, 2, 3, 4],
    "heart_attack": [0, 1],
    "congestive_hf": [0, 1],
    "peripheral_vasc_disease": [0, 1],
}

print()
for col, valid_values in category_rules.items():
    invalid = ~df[col].isin(valid_values)
    print(f"{col} invalid values: {invalid.sum()}")
    if invalid.any():
        df.loc[invalid, col] = np.nan
    df[col] = df[col].astype("Int64")   # keeps whole numbers even if blanks exist


nyha_class invalid values: 0
killip_grade invalid values: 0
heart_attack invalid values: 0
congestive_hf invalid values: 0
peripheral_vasc_disease invalid values: 0


In [16]:
# ============================================================
# STEP 5: CLEAN LVEF (%)
#
# Normal range 55-70%, but low LVEF is common in heart failure,
# so only impossible values (<= 0 or > 100) are removed.
# ============================================================
invalid_lvef = (df["ejection_fraction"] <= 0) | (df["ejection_fraction"] > 100)
print("\nInvalid LVEF values replaced with NaN:", invalid_lvef.sum())
df.loc[invalid_lvef, "ejection_fraction"] = np.nan

df["lvef_normal"] = range_flag(df["ejection_fraction"], 55, 70)


Invalid LVEF values replaced with NaN: 0


In [17]:
# ============================================================
# STEP 6: CLEAN LV END-DIASTOLIC DIAMETER (cm)
#
# Documentation says cm (normal 3.5-5.6), but almost every value
# is 20-100, which only makes sense as mm -> divide by 10.
# After conversion, anything under 2 cm or over 10 cm is not a
# realistic adult LV diameter -> missing.
# ============================================================
lv_mm_values = df["lv_diameter"].between(20, 100)
print("\nLV diameter values converted mm -> cm:", lv_mm_values.sum())
df.loc[lv_mm_values, "lv_diameter"] = df.loc[lv_mm_values, "lv_diameter"] / 10

invalid_lv = (df["lv_diameter"] < 2.0) | (df["lv_diameter"] > 10.0)
print("Invalid LV diameter values replaced with NaN:", invalid_lv.sum())
df.loc[invalid_lv, "lv_diameter"] = np.nan

df["lv_diameter_normal"] = range_flag(df["lv_diameter"], 3.5, 5.6)


LV diameter values converted mm -> cm: 1309
Invalid LV diameter values replaced with NaN: 2


In [18]:

# ============================================================
# STEP 7: FIX MITRAL VELOCITY VALUES (m/s)
#
# Real values run up to about 3.9 m/s, so that range is kept.
# ============================================================
mitral_cols = ["mitral_e_velocity", "mitral_a_velocity"]
e = df["mitral_e_velocity"]
a = df["mitral_a_velocity"]
df["mitral_was_fixed"] = False

# exception: a row where E and A are both in cm/s AND the ratio column proves it
# (patient 728235: 44 / 99 = 0.44, which matches its e_a_ratio)
proven_cm = (e > 5) & (a > 5) & ((e / a - df["e_a_ratio"]).abs() < 0.05)
df.loc[proven_cm, mitral_cols] = df.loc[proven_cm, mitral_cols] / 100
df.loc[proven_cm, "mitral_was_fixed"] = True
print("\nMitral rows converted from cm/s (proven by ratio):", proven_cm.sum())

# everything else above 5 is invalid -> set to missing
for col in mitral_cols:
    invalid = df[col] > 5
    print(f"{col}: {invalid.sum()} set to missing")
    df.loc[invalid, col] = np.nan
    df.loc[invalid, "mitral_was_fixed"] = True



Mitral rows converted from cm/s (proven by ratio): 1
mitral_e_velocity: 10 set to missing
mitral_a_velocity: 6 set to missing


In [19]:
# ============================================================
# STEP 8: FILL + FLAG THE E/A RATIO
#
# The ratio column matches E / A in about 97% of rows that have
# all three, so missing ratios are calculated from E and A.
# Normal range 0.6-1.32.
# ============================================================
calc_ratio = (df["mitral_e_velocity"] / df["mitral_a_velocity"]).round(3)

can_fill = df["e_a_ratio"].isna() & calc_ratio.notna()
df["ea_was_filled"] = can_fill
df.loc[can_fill, "e_a_ratio"] = calc_ratio[can_fill]
print("\ne_a_ratio filled from E / A:", can_fill.sum())

df["ea_normal"] = range_flag(df["e_a_ratio"], 0.6, 1.32)
df["ea_above_2"] = (df["e_a_ratio"] > 2).astype("boolean").mask(df["e_a_ratio"].isna())
df["mitral_e_above_2"] = (df["mitral_e_velocity"] > 2).astype("boolean").mask(df["mitral_e_velocity"].isna())


e_a_ratio filled from E / A: 140


In [20]:

# ============================================================
# STEP 9: CLEAN TRICUSPID RETURN VELOCITY (m/s)
#
# High values can occur in severe cardiac/pulmonary disease,
# so they are kept. Only zero or negative values are removed.
# ============================================================
invalid_tr_velocity = df["tricuspid_velocity"] <= 0
print("\nInvalid tricuspid velocity values:", invalid_tr_velocity.sum())
df.loc[invalid_tr_velocity, "tricuspid_velocity"] = np.nan



Invalid tricuspid velocity values: 0


In [21]:
# ============================================================
# STEP 10: CLEAN TRICUSPID RETURN PRESSURE (mmHg)
#
# High pressures can occur in disease, so they are kept.
# Only zero or negative values are removed.
# ============================================================
invalid_tr_pressure = df["tricuspid_pressure"] <= 0
print("Invalid tricuspid pressure values:", invalid_tr_pressure.sum())
df.loc[invalid_tr_pressure, "tricuspid_pressure"] = np.nan

Invalid tricuspid pressure values: 0


In [22]:
# ============================================================
# STEP 11: ROUND CLEANED MEDICAL VALUES
# ============================================================
measurement_cols = [
    "ejection_fraction",
    "lv_diameter",
    "mitral_e_velocity",
    "mitral_a_velocity",
    "e_a_ratio",
    "tricuspid_velocity",
    "tricuspid_pressure",
]
df[measurement_cols] = df[measurement_cols].round(3)

In [23]:
# ============================================================
# STEP 12: MISSING VALUE REPORT
# ============================================================
missing_report = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": (df.isnull().mean() * 100).round(2),
}).sort_values("missing_percent", ascending=False)

print("\nMissing Values After Cleaning:")
print(missing_report)


Missing Values After Cleaning:
                         missing_count  missing_percent
tricuspid_pressure                1826            90.94
ea_above_2                        1475            73.46
ea_normal                         1475            73.46
e_a_ratio                         1475            73.46
mitral_a_velocity                 1464            72.91
lvef_normal                       1373            68.38
ejection_fraction                 1373            68.38
tricuspid_velocity                1218            60.66
mitral_e_velocity                 1038            51.69
mitral_e_above_2                  1038            51.69
lv_diameter                        699            34.81
lv_diameter_normal                 699            34.81
nyha_class                           0             0.00
hf_type                              0             0.00
peripheral_vasc_disease              0             0.00
congestive_hf                        0             0.00
mitral_was_fixed

In [24]:
# ============================================================
# STEP 13: DISPLAY CLEANED MEDICAL DATA
# ============================================================
print("\nCleaned medical values:")
print(df[measurement_cols].describe().round(2))


Cleaned medical values:
       ejection_fraction  lv_diameter  mitral_e_velocity  mitral_a_velocity  \
count             635.00      1309.00             970.00             544.00   
mean               50.68         5.32               1.06               0.83   
std                13.22         1.07               0.46               0.32   
min                 5.00         2.20               0.03               0.06   
25%                41.00         4.50               0.76               0.58   
50%                51.00         5.30               1.00               0.82   
75%                61.00         6.00               1.26               1.04   
max                82.00         8.80               3.90               2.80   

       e_a_ratio  tricuspid_velocity  tricuspid_pressure  
count     533.00              790.00              182.00  
mean        1.29                2.99               35.91  
std         1.18                0.62               13.70  
min         0.06           

In [25]:
# ============================================================
# STEP 14: RESET INDEX + SAVE
# ============================================================
df = df.reset_index(drop=True)
df.to_csv("cardiac_complications_cleaned.csv", index=False)

print("\nCleaning completed successfully.")
print("Final dataset shape:", df.shape)


Cleaning completed successfully.
Final dataset shape: (2008, 21)
